In [3]:
import math
import pathlib

1. Parsing the System of Equations (1 point)

In [13]:
def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    system=path.read_text()
    lines=system.splitlines()
    free_terms=[]
    system_matrix=[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    for index,line in enumerate(lines):
        split_equal=line.split('=')
        split_equal[0]=split_equal[0]
        free_terms.append(float(split_equal[1]))
        left_part=split_equal[0].replace(' ','')
        split_x=left_part.split('x')
        if len(split_x) == 1:
            system_matrix[index][0] = 0.0
            split_y=split_x[0].split('y')
        else:
            system_matrix[index][0]=1.0 if split_x[0] in ('','+') else (-1.0 if split_x[0]=='-' else float(split_x[0]))
            split_y=split_x[1].split('y')
        if len(split_y) == 1:
            system_matrix[index][1] = 0.0
            split_z=split_y[0].split('z')
        else:
            system_matrix[index][1]=1.0 if split_y[0] in ('','+') else (-1.0 if split_y[0]=='-' else float(split_y[0]))
            split_z=split_y[1].split('z')
        if len(split_z) == 1:
            system_matrix[index][2] = 0.0
        else:
            system_matrix[index][2] = 1.0 if split_z[0] in ('', '+') else (-1.0 if split_z[0] == '-' else float(split_z[0]))
    return system_matrix,free_terms

notebook_dir = pathlib.Path().resolve()
path = notebook_dir / "system.txt"
A,B=load_system(pathlib.Path(path))
print(f"{A=} {B=}")

A=[[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]] B=[5.0, 6.0, 7.0]


2. Matrix and Vector Operations (5 points)


2.1 Determinant

In [5]:
def determinant(matrix: list[list[float]]) -> float:
    a11=matrix[0][0]*(matrix[1][1]*matrix[2][2]-matrix[1][2]*matrix[2][1])
    a12=matrix[0][1]*(matrix[1][0]*matrix[2][2]-matrix[1][2]*matrix[2][0])
    a13=matrix[0][2]*(matrix[1][0]*matrix[2][1]-matrix[1][1]*matrix[2][0])
    return a11-a12+a13
print(f"{determinant(A)=}")

determinant(A)=14.0


2.2 Trace

In [6]:
def trace(matrix: list[list[float]]) -> float:
    return sum(matrix[i][i] for i in range(len(matrix)))

print(f"{trace(A)=}")

trace(A)=3.0


2.3 Vector norm

In [7]:
def norm(vector: list[float]) -> float:
    return math.sqrt(sum(x * x for x in vector))

print(f"{norm(B)=}")

norm(B)=10.488088481701515


2.4. Transpose of matrix


In [8]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    return [[ matrix[row][col] for row in range(len(matrix)) ] for col in range(len(matrix[0]))]

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


2.5. Matrix-vector multiplication

In [9]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    product=[0.0 for _ in range(len(vector))]
    for row in range(len(matrix)):
        for col in range(len(matrix[row])):
            product[row]+=matrix[row][col]*vector[col]
    return product

print(f"{multiply(A, B)=}")

multiply(A, B)=[21.0, 27.0, 35.0]


3. Solving using Cramer's Rule (1 point)


In [10]:
def solve_cramer(matrix: list[list[float]],vector: list[float]) -> list[float]:
   solution=[]
   delta=determinant(matrix)
   if delta==0.0:
       raise ValueError("The system is not uniquely solvable system")
   for i in range(3):
    for index in range(len(vector)):
        matrix[index][i],vector[index]=vector[index],matrix[index][i]
    solution.append(determinant(matrix)/delta)
    for index in range(len(vector)):
        matrix[index][i], vector[index] = vector[index], matrix[index][i]

   return solution

print(f"{solve_cramer(A, B)=}")


solve_cramer(A, B)=[0.35714285714285715, 2.0714285714285716, 1.9285714285714286]


4. Solving using Inversion

In [11]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
   minor_matrix=[]
   for row_idx,row in enumerate(matrix):
       if row_idx==i:
           continue
       minor_row=[]
       for col_idx,col in enumerate(row):
           if col_idx==j:
               continue
           minor_row.append(col)
       minor_matrix.append(minor_row)
   return minor_matrix

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    n=len(matrix)
    cofactor_matrix=[]
    for i in range(n):
        cofactor_row=[]
        for j in range(n):
            minor_matrix=minor(matrix,i,j)
            delta_minor=(-1)**(i+j)
            delta_minor*=(minor_matrix[0][0]*minor_matrix[1][1]-
                          minor_matrix[0][1]*minor_matrix[1][0])
            cofactor_row.append(delta_minor)
        cofactor_matrix.append(cofactor_row)
    return cofactor_matrix


def adjoint(matrix: list[list[float]]) -> list[list[float]]:
     return cofactor(transpose(matrix))

def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    delta=determinant(matrix)
    if delta==0.0:
       raise ValueError("The system is not uniquely solvable system")
    solution=multiply(adjoint(matrix),vector)
    return [val/delta for val in solution]

print(f"{solve(A, B)=}")

solve(A, B)=[0.35714285714285715, 2.0714285714285716, 1.9285714285714286]
